### Data Ingestion

In [ ]:
from langchain_core.documents import Document
document = Document(page_content="this is the main text content i m using to create RAG",
                    metadata={
                        "source" : "anukalp.24",
                        "pages" : 1,
                        "author_name": "Anukalp Agarwal",
                        "data_created" : "2025-01-01"
                    })

#metadata is very useful as when we do similarity search we can also apply filters.



In [2]:
#Text Loader , The First Step

# Text loader is jsut a configured tool
from langchain_community.document_loaders import TextLoader

loader = TextLoader("../data/agentic_ai.txt" , encoding="utf-8")
document = loader.load()
# load() is the qactionwhich actually  scans file and creates a document structure
print(document)





[Document(metadata={'source': '../data/agentic_ai.txt'}, page_content='write a  paragraph agenitc ai ai agents each 1 1 para\n\nAgentic AI\nAgentic AI refers to AI systems designed to pursue goals with a degree of autonomy — rather than simply responding to a single prompt and stopping, an agentic system can plan multi-step actions, make decisions along the way, use tools, and adapt its approach based on results, all with minimal human intervention at each step. Instead of just generating text once, an agentic AI might break a task into sub-tasks, decide which tool or API to call, evaluate the outcome, and loop back to try a different approach if something fails — behaving less like a one-shot responder and more like an autonomous problem-solver working toward an end goal.')]


In [5]:
###directory loader
# it can load multiple files in one go

from langchain_community.document_loaders import PyPDFLoader , PyMuPDFLoader , DirectoryLoader

directoryLoader = DirectoryLoader("../data" , glob="**/*.pdf" , loader_cls=PyMuPDFLoader , show_progress=False)
document = directoryLoader.load()
print(document)

[Document(metadata={'producer': 'ReportLab PDF Library - (opensource)', 'creator': 'anonymous', 'creationdate': '2026-08-11T11:50:08+00:00', 'source': '..\\data\\Anukalp_Agarwal_Resume_Exact_Final.pdf', 'file_path': '..\\data\\Anukalp_Agarwal_Resume_Exact_Final.pdf', 'total_pages': 1, 'format': 'PDF 1.3', 'title': 'untitled', 'author': 'anonymous', 'subject': 'unspecified', 'keywords': '', 'moddate': '2026-08-11T11:50:08+00:00', 'trapped': '', 'modDate': "D:20260811115008+00'00'", 'creationDate': "D:20260811115008+00'00'", 'page': 0}, page_content=''), Document(metadata={'producer': 'PDFsharp 6.1.1', 'creator': 'PDFsharp 6.1.1 (www.pdfsharp.net)', 'creationdate': '2026-05-28T21:01:52+05:30', 'source': '..\\data\\healthrecords.pdf', 'file_path': '..\\data\\healthrecords.pdf', 'total_pages': 13, 'format': 'PDF 1.7', 'title': '', 'author': '', 'subject': '', 'keywords': '', 'moddate': '2026-05-28T21:01:52+05:30', 'trapped': '', 'modDate': "D:20260528210152+05'30'", 'creationDate': "D:2026

In [ ]:
from langchain_community.document_loaders import PyPDFLoader , PyMuPDFLoader , DirectoryLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
import streamlit as st
st.title("My RAG App")
def pdf(pdf_directory):
 AllDocuments = []
 pdf_dir= Path(pdf_directory) 
    #  path converts the plain string pdf directory into a path obejct so u can call glob()
    # print(type(pdf_dir))
    # print(pdf_dir)

# we need to wrap it in a list cause then only we can see the file names
 pdf_files = list(pdf_dir.glob("**/*.pdf"))
       # .glob() is a method that only exists on Path objects thats why we convert the directory into path object
    # print(pdf_files)  it will return this [
    # WindowsPath('../data/report.pdf'),
    # WindowsPath('../data/notes.pdf')
# ]

 try:
     for pdf_file in pdf_files:
         print(pdf_file.name)
         loader = PyMuPDFLoader(str(pdf_file))       # we need to conevrt the apth obj into stricng before sending it to PyMuPDFLoader
         document = loader.load()

         for doc in document:
          doc.metadata["source_file"] = pdf_file.name

         AllDocuments.extend(document)
     text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000 , chunk_overlap=200 , separators=["\n\n" , "\n" , " " , ""] , length_function=len)   #  its just a tool/rules
     chunks = text_splitter.split_documents(AllDocuments)   #  it actually creates chunks
 
# sentence tranformers creates the embedding        s
     embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
 
     vectorDB = Chroma.from_documents(documents=chunks, embedding=embeddings , collection_name="pdf_documents") # stores in vector db
    
    # vector db stores the  original chunks , embeddings and meta data which of that chunk

     retriever = vectorDB.as_retriever(search_kwargs={"k" : 3})
     question = st.text_input("Ask a question")
     if question:
        st.write("you asked" , question)
    


     print(document)
     return vectorDB
 except Exception as error:
     print(error)
    #  continue

# ex how documents are stored inside a list:-
# Resume = 1 page
# Health = 5 pages
# then after both:
# AllDocuments = 6 Document objects
# see docuemnt is one array which will consit of many docuemnt  as per page
pdf("../data") # function calling

In [ ]:
from fastapi.responses import JSONResponse
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader, DirectoryLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from langchain_groq import ChatGroq
from dotenv import load_dotenv

async def searchController(files):

    try:
           AllDocuments = []
    pdf_dir = Path(files)
    #  path converts the plain string pdf directory into a path obejct so u can call glob()
    # print(type(pdf_dir)) 
    # print(pdf_dir)

    # we need to wrap it in a list cause then only we can see the file names
    pdf_files = list(pdf_dir.glob("**/*.pdf"))
    # .glob() is a method that only exists on Path objects thats why we convert the directory into path object
    # print(pdf_files)  it will return this [
    # WindowsPath('../data/report.pdf'),
    # WindowsPath('../data/notes.pdf')
    # ]

    try:
        for pdf_file in pdf_files:
            print(pdf_file.name)
            loader = PyMuPDFLoader(str(pdf_file))  # we need to conevrt the apth obj into stricng before sending it to PyMuPDFLoader
            document = loader.load()

            for doc in document:
                doc.metadata["source_file"] = pdf_file.name

            AllDocuments.extend(document)

        text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=100, separators=["\n\n", "\n", ". ", " ", ""], length_function=len)  # its just a tool/rules

        chunks = text_splitter.split_documents(AllDocuments)  # it actually creates chunks and return a array of chunks

        #      A chunk internally contains two main things:

        # python
        # Document(
        #     page_content="...",   # the actual text
        #     metadata={...}         # extra info about where this text came from
        # )
        # sentence tranformers creates the embedding        s
        embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

        vectorDB = Chroma.from_documents(documents=chunks, embedding=embeddings, collection_name="pdf_documents")  # stores in vector db

        # vector db stores the  original chunks , embeddings and meta data which of that chunk
        retriever = vectorDB.as_retriever(search_kwargs={"k": 5})  # return three most relevant chunks

        question = st.text_input("Ask a Question")
        if question:
            docs = retriever.invoke(question)    # this line says hey retriever go find the relevant  chunks and it will return those chunks
                        # # it will return stred text knwos as page content which is original chunk text  and meta data
            

            if not docs:
                 
                return JSONResponse(status_code=400 , content={"message" : "No relevant information found in the document"})

               
            context = "\n\n".join([doc.page_content for doc in docs])
            prompt = f"""You are a precise, factual assistant that answers questions strictly based on the provided document context. You never use outside knowledge, even if you know the answer.
    
# Rules:
# 1. Answer ONLY using information explicitly stated in the context below.
# 2. If the context does not contain enough information to answer, respond exactly with: "The document doesn't contain enough information to answer this question."
# 3. Do not guess, infer beyond what's written, or add information not present in the context.
# 4. Keep your answer clear and concise — no unnecessary preamble like "Based on the context provided."
# 5. If helpful, quote or closely reference the specific part of the context that supports your answer.


# # context:{context}
# # Question:{question}
# #         """
#             response = llm.invoke(prompt)
#           

            print(document)
     except Exception as error:
      return JSONResponse(status_code=500 , content={"message" : "Internal server error "})
       
      print(error)
    #   continue
#     # # ex how documents are stored inside a list:-
#     # # Resume = 1 page
#     # # Health = 5 pages
#     # # then after both:
#     # # AllDocuments = 6 Document objects
#     # # see docuemnt is one array which will consit of many docuemnt  as per page


